# Градиентный бустинг для бинарной классификации

Сравниваются собственная реализация `Gradient Boosting`
и `XGBoost` на задаче классификации опухолей молочной железы.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "gradient_boosting"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


 
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Библиотеки подключены")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"xgboost: {xgb.__version__}")

In [ ]:
# Загрузка данных и подготовка признаков
data = load_breast_cancer()
X = data.data
y = data.target

print('Набор данных: Breast Cancer Wisconsin')
print(f'Объектов: {X.shape[0]}, признаков: {X.shape[1]}')
print(f'Classes: {data.target_names}')
print(f'Распределение классов: {np.bincount(y)}')
print(f'Доля положительного класса: {y.mean():.3f}')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f'\nРазмер train: {X_train.shape[0]}')
print(f'Размер validation: {X_val.shape[0]}')
print(f'Распределение классов в train: {np.bincount(y_train)}')
print(f'Распределение классов в validation: {np.bincount(y_val)}')

In [ ]:
# Реализация MyGradientBoosting
class MyGradientBoosting:
    """
    Gradient Boosting для бинарной классификации с log-loss.

    На каждой итерации:
      1. считаем отрицательный градиент (остатки) = y_true - sigmoid(F)
      2. обучаем DecisionTreeRegressor на остатках
      3. обновляем оценки: F += learning_rate * tree.predict(X)

    Предсказания получаем, применяя sigmoid к сырым логитам и
    порог 0.5.
    """

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.init_prediction = None
        self.train_losses = []
        self.val_losses = []

    @staticmethod
    def _sigmoid(x):
        """Численно устойчивая сигмоида."""
        return np.where(
            x >= 0,
            1.0 / (1.0 + np.exp(-x)),
            np.exp(x) / (1.0 + np.exp(x)),
        )

    @staticmethod
    def _log_loss(y_true, raw_scores):
        """Binary cross-entropy по сырым logit-оценкам."""
        probs = MyGradientBoosting._sigmoid(raw_scores)
        probs = np.clip(probs, 1e-15, 1 - 1e-15)
        return -np.mean(y_true * np.log(probs) + (1 - y_true) * np.log(1 - probs))

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """Обучает ансамбль градиентного бустинга."""
        if X_val is None and y_val is not None:
            raise ValueError("y_val was provided but X_val is None")
        if X_val is not None and y_val is None:
            raise ValueError("X_val was provided but y_val is None")

        y_train = np.asarray(y_train, dtype=np.float64)

        # Сбрасываем внутреннее состояние, чтобы повторный fit() не копил старые деревья.
        self.trees = []
        self.train_losses = []
        self.val_losses = []

        mean_y = np.clip(y_train.mean(), 1e-15, 1 - 1e-15)
        self.init_prediction = np.log(mean_y / (1.0 - mean_y))

        F_train = np.full(len(y_train), self.init_prediction, dtype=np.float64)
        if X_val is not None:
            y_val = np.asarray(y_val, dtype=np.float64)
            F_val = np.full(len(y_val), self.init_prediction, dtype=np.float64)

        self.train_losses.append(self._log_loss(y_train, F_train))
        if X_val is not None:
            self.val_losses.append(self._log_loss(y_val, F_val))

        for i in range(self.n_estimators):
            residuals = y_train - self._sigmoid(F_train)

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=42 + i,
            )
            tree.fit(X_train, residuals)
            self.trees.append(tree)

            F_train += self.learning_rate * tree.predict(X_train)
            if X_val is not None:
                F_val += self.learning_rate * tree.predict(X_val)

            self.train_losses.append(self._log_loss(y_train, F_train))
            if X_val is not None:
                self.val_losses.append(self._log_loss(y_val, F_val))

        return self

    def predict_proba(self, X):
        """Возвращает вероятность класса 1 для каждого объекта."""
        F = np.full(len(X), self.init_prediction, dtype=np.float64)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        prob_pos = self._sigmoid(F)
        return np.column_stack([1 - prob_pos, prob_pos])

    def predict(self, X, threshold=0.5):
        """Возвращает бинарные предсказания."""
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


print("Класс MyGradientBoosting готов")

In [ ]:
# Обучаем свою реализацию и собираем метрики
N_ESTIMATORS = 150
LEARNING_RATE = 0.05
MAX_DEPTH = 3

my_gb = MyGradientBoosting(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
)

my_start = time.perf_counter()
my_gb.fit(X_train_scaled, y_train, X_val=X_val_scaled, y_val=y_val)
my_fit_time = time.perf_counter() - my_start

y_train_pred_my = my_gb.predict(X_train_scaled)
y_val_pred_my = my_gb.predict(X_val_scaled)
y_val_proba_my = my_gb.predict_proba(X_val_scaled)[:, 1]

train_acc_my = accuracy_score(y_train, y_train_pred_my)
val_acc_my = accuracy_score(y_val, y_val_pred_my)
final_train_loss = my_gb.train_losses[-1]
final_val_loss = my_gb.val_losses[-1]

print("=== Результаты MyGradientBoosting ===")
print(f"Estimators: {N_ESTIMATORS}, LR: {LEARNING_RATE}, Max depth: {MAX_DEPTH}")
print(f"Время обучения : {my_fit_time:.4f} сек")
print(f"Точность train : {train_acc_my:.4f}")
print(f"Точность val   : {val_acc_my:.4f}")
print(f"Train log-loss : {final_train_loss:.4f}")
print(f"Val log-loss   : {final_val_loss:.4f}")
print(f"Разрыв train-val: {train_acc_my - val_acc_my:.4f}")

if val_acc_my >= 0.70:
    print(f"\nПорог пройден: val accuracy {val_acc_my:.4f} >= 0.70")
else:
    print(f"\nПорог не пройден: val accuracy {val_acc_my:.4f} < 0.70. Нужна подстройка.")

In [ ]:
# Обучаем XGBoost для сравнения
xgb_model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    verbosity=0,
)

eval_set = [(X_train_scaled, y_train), (X_val_scaled, y_val)]

xgb_start = time.perf_counter()
xgb_model.fit(
    X_train_scaled,
    y_train,
    eval_set=eval_set,
    verbose=False,
)
xgb_fit_time = time.perf_counter() - xgb_start

evals_result = xgb_model.evals_result()
xgb_train_loss = evals_result["validation_0"]["logloss"]
xgb_val_loss = evals_result["validation_1"]["logloss"]

y_train_pred_xgb = xgb_model.predict(X_train_scaled)
y_val_pred_xgb = xgb_model.predict(X_val_scaled)

train_acc_xgb = accuracy_score(y_train, y_train_pred_xgb)
val_acc_xgb = accuracy_score(y_val, y_val_pred_xgb)

print("=== Результаты XGBoost ===")
print(f"Estimators: {N_ESTIMATORS}, LR: {LEARNING_RATE}, Max depth: {MAX_DEPTH}")
print(f"Время обучения : {xgb_fit_time:.4f} сек")
print(f"Точность train : {train_acc_xgb:.4f}")
print(f"Точность val   : {val_acc_xgb:.4f}")
print(f"Train log-loss : {xgb_train_loss[-1]:.4f}")
print(f"Val   log-loss : {xgb_val_loss[-1]:.4f}")
print(f"Разрыв train-val: {train_acc_xgb - val_acc_xgb:.4f}")

In [ ]:
# Графики потерь
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gradient Boosting: потери по итерациям', fontsize=14, fontweight='bold')

# Левая панель: MyGradientBoosting
ax1 = axes[0]
iters_my = range(len(my_gb.train_losses))
ax1.plot(iters_my, my_gb.train_losses, label='Train loss', color='steelblue', linewidth=1.8)
ax1.plot(iters_my, my_gb.val_losses,   label='Val loss',   color='coral',     linewidth=1.8, linestyle='--')

# Подписываем финальные значения
ax1.annotate(
    f'{my_gb.train_losses[-1]:.4f}',
    xy=(len(my_gb.train_losses) - 1, my_gb.train_losses[-1]),
    xytext=(-35, 8), textcoords='offset points',
    fontsize=9, color='steelblue'
)
ax1.annotate(
    f'{my_gb.val_losses[-1]:.4f}',
    xy=(len(my_gb.val_losses) - 1, my_gb.val_losses[-1]),
    xytext=(-35, -16), textcoords='offset points',
    fontsize=9, color='coral'
)

ax1.set_xlabel('Итерация', fontsize=11)
ax1.set_ylabel('Log-Loss', fontsize=11)
ax1.set_title('MyGradientBoosting', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Правая панель: XGBoost
ax2 = axes[1]
iters_xgb = range(1, len(xgb_train_loss) + 1)
ax2.plot(iters_xgb, xgb_train_loss, label='Train loss', color='steelblue', linewidth=1.8)
ax2.plot(iters_xgb, xgb_val_loss,   label='Val loss',   color='coral',     linewidth=1.8, linestyle='--')

ax2.annotate(
    f'{xgb_train_loss[-1]:.4f}',
    xy=(len(xgb_train_loss), xgb_train_loss[-1]),
    xytext=(-35, 8), textcoords='offset points',
    fontsize=9, color='steelblue'
)
ax2.annotate(
    f'{xgb_val_loss[-1]:.4f}',
    xy=(len(xgb_val_loss), xgb_val_loss[-1]),
    xytext=(-35, -16), textcoords='offset points',
    fontsize=9, color='coral'
)

ax2.set_xlabel('Итерация', fontsize=11)
ax2.set_ylabel('Log-Loss', fontsize=11)
ax2.set_title('XGBoost', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gb_loss_curves.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('График сохранён: gb_loss_curves.png')

In [ ]:
# Сравнение точности и короткая сводка
results = pd.DataFrame({
    "Модель": ["MyGradientBoosting", "XGBoost"],
    "Время обучения, с": [my_fit_time, xgb_fit_time],
    "Train Accuracy": [train_acc_my, train_acc_xgb],
    "Val Accuracy": [val_acc_my, val_acc_xgb],
    "Train Log-Loss": [final_train_loss, xgb_train_loss[-1]],
    "Val Log-Loss": [final_val_loss, xgb_val_loss[-1]],
    "Разрыв train-val": [train_acc_my - val_acc_my, train_acc_xgb - val_acc_xgb],
})
results = results.round(4)

print("=== Итоговое сравнение ===")
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Сравнение моделей: MyGradientBoosting и XGBoost", fontsize=13, fontweight="bold")

models = ["MyGB", "XGBoost"]
x = np.arange(len(models))
width = 0.35

ax1 = axes[0]
bars1 = ax1.bar(x - width / 2, [train_acc_my, train_acc_xgb], width, label="Train", color="steelblue", alpha=0.85)
bars2 = ax1.bar(x + width / 2, [val_acc_my, val_acc_xgb], width, label="Validation", color="coral", alpha=0.85)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)

ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=11)
ax1.set_ylabel("Accuracy", fontsize=11)
ax1.set_title("Точность", fontsize=12)
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.08)
ax1.axhline(y=0.70, color="red", linestyle=":", alpha=0.6)
ax1.grid(True, alpha=0.3, axis="y")

ax2 = axes[1]
bars3 = ax2.bar(x - width / 2, [final_train_loss, xgb_train_loss[-1]], width, label="Train", color="steelblue", alpha=0.85)
bars4 = ax2.bar(x + width / 2, [final_val_loss, xgb_val_loss[-1]], width, label="Validation", color="coral", alpha=0.85)

for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
for bar in bars4:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)

ax2.set_xticks(x)
ax2.set_xticklabels(models, fontsize=11)
ax2.set_ylabel("Log-Loss", fontsize=11)
ax2.set_title("Log-Loss (меньше лучше)", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis="y")

ax3 = axes[2]
bars5 = ax3.bar(models, [my_fit_time, xgb_fit_time], color=["steelblue", "coral"], alpha=0.85)
for bar in bars5:
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
ax3.set_ylabel("Секунды", fontsize=11)
ax3.set_title("Время обучения", fontsize=12)
ax3.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gb_comparison.png', dpi=120, bbox_inches="tight")
plt.show()

speed_ratio = my_fit_time / xgb_fit_time if xgb_fit_time > 0 else float("inf")

print("\n=== Что получилось ===")
print(f"MyGradientBoosting, val accuracy: {val_acc_my:.4f}")
print(f"XGBoost, val accuracy:          {val_acc_xgb:.4f}")
print(f"MyGradientBoosting, время:      {my_fit_time:.4f} сек")
print(f"XGBoost, время:                 {xgb_fit_time:.4f} сек")
print(f"Отношение скоростей (MyGB / XGB): {speed_ratio:.2f}x")
print()
print("Наблюдения:")
print("  - В своей реализации весь цикл бустинга виден явно: остатки, дерево, аддитивное обновление.")
print("  - XGBoost обычно выигрывает по log-loss за счёт более сильной инженерной обвязки и регуляризации.")
print("  - Обе модели проходят порог 70% по validation accuracy, так что базовая реализация работает корректно.")
print("  - Время обучения измеряется явно, поэтому сравнение не на глаз, а по числам.")